# Task 1. Clone Repository và Khám Phá File

## Mục tiêu

Task này chuẩn bị dữ liệu đầu vào cho pipeline Code Property Graph (CPG). Nhóm sử dụng CLI chính thức của dự án để clone repository mục tiêu `huggingface/transformers-pr-agent` và chạy quy trình discovery xác định danh sách file eligible.

Mọi thống kê được thực hiện bằng cách đọc file manifest được tạo bởi core service, đảm bảo tính nhất quán tuyệt đối giữa báo cáo và code dự án.


## Clone Repository

Repository được clone shallow bằng `--depth 1` để lấy snapshot mới nhất mà không tải toàn bộ lịch sử Git. Nếu thư mục đã tồn tại, notebook không clone lại mà dùng bản local hiện có.

In [1]:
import os
import json
import subprocess
from pathlib import Path

# Resolve PROJECT_ROOT using Git
PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

SOURCE_REPOSITORY = PROJECT_ROOT / "workspace/source/transformers-pr-agent"
MANIFEST_PATH = PROJECT_ROOT / "artifacts/manifests/source-files.jsonl"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SOURCE_REPOSITORY:", SOURCE_REPOSITORY)
print("MANIFEST_PATH:", MANIFEST_PATH)


PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming
SOURCE_REPOSITORY: /home/phat/AI_Project/lab04-cpg-streaming/workspace/source/transformers-pr-agent
MANIFEST_PATH: /home/phat/AI_Project/lab04-cpg-streaming/artifacts/manifests/source-files.jsonl


## Khám Phá Cấu Trúc Repository

Cell dưới đây in cây thư mục rút gọn ở mức cao. Các thư mục build, cache và `.git` được bỏ qua để phần hiển thị tập trung vào cấu trúc nguồn.

In [2]:
# Execute CLI clone command
print("Executing clone-source CLI...")
subprocess.run(
    ["uv", "run", "lab04", "clone-source"],
    check=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)

# Verification using Git
remote_url = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPOSITORY), "remote", "get-url", "origin"],
    text=True,
).strip()
commit_hash = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPOSITORY), "rev-parse", "HEAD"],
    text=True,
).strip()
is_shallow = subprocess.check_output(
    ["git", "-C", str(SOURCE_REPOSITORY), "rev-parse", "--is-shallow-repository"],
    text=True,
).strip()

print("Remote URL:", remote_url)
print("Commit SHA:", commit_hash)
print("Shallow repository:", is_shallow)


Executing clone-source CLI...


Cloning target source repository...
Cloned successfully to: workspace/source/transformers-pr-agent
Commit SHA: 458c957fa1e8851825cd799f5d030876f0644194
Remote URL: https://github.com/huggingface/transformers-pr-agent.git
Commit SHA: 458c957fa1e8851825cd799f5d030876f0644194
Shallow repository: true


## Thống Kê File Python Theo Thư Mục

Trước khi chọn phạm vi phân tích, nhóm đếm toàn bộ file `.py` theo thư mục cấp cao nhất. Bước này giúp phân biệt mã nguồn chính với test, ví dụ, benchmark và script phụ trợ.

In [3]:
# Run discovery CLI command
print("Executing discover CLI...")
subprocess.run(
    ["uv", "run", "lab04", "discover", "--scope", "final", "--manifest", str(MANIFEST_PATH)],
    check=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)


Executing discover CLI...


Executing discovery phase (scope=final, manifest=/home/phat/AI_Project/lab04-cpg-streaming/artifacts/manifests/source-files.jsonl)...


Discovery phase completed. Eligible files: 2779


CompletedProcess(args=['uv', 'run', 'lab04', 'discover', '--scope', 'final', '--manifest', '/home/phat/AI_Project/lab04-cpg-streaming/artifacts/manifests/source-files.jsonl'], returncode=0)

## Chọn Phạm Vi Phân Tích Chính

Nhóm chọn các file Python trong `src/` làm input chính cho Parser Service. Các file `setup.py`, `conftest.py` và thư mục cache không được đưa vào parse vì không đại diện cho mã nguồn thư viện cần phân tích.

In [4]:
# Read and summarize manifest generated by CLI/service
eligible_files = []
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        eligible_files.append(record)

print("Manifest contains eligible python files after applying scope and filters.")
print("Total eligible files:", len(eligible_files))

# Distribution by top-level folder
from collections import Counter
by_top_level = Counter(
    Path(record["file_path"]).parts[0] for record in eligible_files
)
print("\nPython files by top-level path:")
for name, count in by_top_level.most_common():
    print(f"{name}: {count}")

# Compute size statistics using built-in statistics module
sizes = [record["size_bytes"] for record in eligible_files]
import statistics
print("\nFile size statistics (bytes):")
print("Min size:", min(sizes))
print("Max size:", max(sizes))
print("Mean size:", statistics.mean(sizes))
print("Median size:", statistics.median(sizes))

# Read repository metadata from manifest
repo_ids = {record["repository_id"] for record in eligible_files}
commit_shas = {record["commit_sha"] for record in eligible_files}
print(f"\nRepository ID: {list(repo_ids)[0]}")
print(f"Commit SHA in manifest: {list(commit_shas)[0]}")

print("\nFirst 20 eligible files:")
for record in eligible_files[:20]:
    print(record["file_path"])


Manifest contains eligible python files after applying scope and filters.
Total eligible files: 2779

Python files by top-level path:
src: 2779

File size statistics (bytes):
Min size: 0
Max size: 267972
Mean size: 17726.211227060092
Median size: 8825

Repository ID: huggingface/transformers-pr-agent
Commit SHA in manifest: 458c957fa1e8851825cd799f5d030876f0644194

First 20 eligible files:
src/transformers/__init__.py
src/transformers/_typing.py
src/transformers/activations.py
src/transformers/audio_utils.py
src/transformers/backbone_utils.py
src/transformers/cache_utils.py
src/transformers/cli/__init__.py
src/transformers/cli/add_new_model_like.py
src/transformers/cli/chat.py
src/transformers/cli/download.py
src/transformers/cli/serve.py
src/transformers/cli/serving/__init__.py
src/transformers/cli/serving/chat_completion.py
src/transformers/cli/serving/completion.py
src/transformers/cli/serving/model_manager.py
src/transformers/cli/serving/response.py
src/transformers/cli/serving/ser

In [5]:
# Task 1 Verification Assertions
assert SOURCE_REPOSITORY.exists(), "Source repository path does not exist"
assert (SOURCE_REPOSITORY / ".git").exists(), "Not a git repository"
assert is_shallow == "true", "Repository is not shallow"
assert MANIFEST_PATH.exists(), "Manifest file does not exist"
assert len(eligible_files) > 0, "Manifest is empty"

# Verify paths are relative POSIX paths and no duplicates
seen_paths = set()
for record in eligible_files:
    path_str = record["file_path"]
    assert not Path(path_str).is_absolute(), f"Path must be relative: {path_str}"
    assert "\\" not in path_str, f"Path must use forward slashes: {path_str}"
    assert path_str not in seen_paths, f"Duplicate file path detected: {path_str}"
    seen_paths.add(path_str)
    
    # Check metadata alignment
    assert record["commit_sha"] == commit_hash, "Commit SHA mismatch in manifest"
    assert record["repository_id"] == "huggingface/transformers-pr-agent", "Repo ID mismatch"

print("Task 1 verification assertions PASSED successfully!")


Task 1 verification assertions PASSED successfully!


## Kết Quả

Kết quả quan trọng của Task 1 là xác định được input ổn định cho các task sau:

- Repository mục tiêu: `huggingface/transformers-pr-agent`.
- Commit được ghi lại để kết quả parse có thể truy vết.
- Phạm vi parse chính: `transformers-pr-agent/src`.
- File test, ví dụ, benchmark, docs và script phụ trợ không nằm trong phạm vi parse chính.

Cách chọn này giúp Parser Service xử lý dữ liệu đại diện cho mã nguồn thư viện, đồng thời giảm dung lượng event khi chạy pipeline streaming.

## Reflection

Task 1 cho thấy bước khảo sát dữ liệu là cần thiết trước khi viết parser. Nếu parse toàn bộ repository, số lượng file phụ trợ và test sẽ làm dữ liệu CPG nhiều nhiễu hơn. Việc giới hạn vào `src/` giúp pipeline nhỏ gọn, dễ kiểm chứng và phù hợp với mục tiêu lab: chứng minh luồng xử lý incremental từ mã nguồn Python sang Kafka, Neo4j và MongoDB.